# **Mini Transformer Language Model From Scratch**

This project implements a small transformer-based language model
using PyTorch internals.

The goal is to understand how modern LLM architectures work by
implementing the core components manually:

* Token embedding
* Positional encoding
* Multi-head self-attention
* Transformer blocks
* KV caching
* Autoregressive text generation

The model is intentionally small (4 layers) to make experimentation easy.

## **Preprocessing**

Importing libraries needed and define hyperparameters for the model

In [76]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

In [77]:
# Define hyperparameters
vocab_size = 5000
d_model = 256
num_heads = 8
num_layers = 4
seq_len = 128
ffn_hidden = 4 * d_model
dropout = 0.1

**d_model** is embedding dimension\
**num_heads** is number of attention heads

## **Token Embedding**

In [93]:
base_model.model.embed_tokens.state_dict()["weight"]

tensor([[ 0.0391,  0.0142, -0.0154,  ..., -0.0339,  0.0147,  0.0261],
        [ 0.0112,  0.0142, -0.0121,  ..., -0.0060, -0.0133,  0.0430],
        [-0.0271, -0.0248,  0.0176,  ..., -0.0281,  0.0031,  0.0249],
        ...,
        [-0.0130,  0.0016, -0.0002,  ...,  0.0050,  0.0014, -0.0009],
        [-0.0130,  0.0016, -0.0002,  ...,  0.0050,  0.0014, -0.0009],
        [-0.0130,  0.0016, -0.0002,  ...,  0.0050,  0.0014, -0.0009]])

In [54]:
class TokenEmbedding(nn.Module):
    """
    Transform token ids into dense vector
    """
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x)

In [103]:
base_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (rotary_emb):

## **Positional Embedding**

In [55]:
class PositionalEncoding(nn.Module):
    """
    Adds position information so the transformer knows token order.
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) *
            (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## **Scaled Dot Product Attention**

$Attention(Q,K,V) = softmax(\frac{QKᵀ}{√d_k}) V$

In [69]:
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attention = torch.softmax(scores, dim=-1)
    output = torch.matmul(attention, v)

    return output, attention

## **Multi-Head Attention (Manual Implementation)**

In [70]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        out, attn = scaled_dot_product_attention(q, k, v)
        out = out.transpose(1,2).contiguous().view(B, T, C)

        return self.out_proj(out), attn

## **Feed Forward Network**

In [71]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)

## **Transformers Block**

In [72]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_hidden):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, ffn_hidden)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, attn_weights = self.attn(x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)

        return x, attn_weights

Residual connections stabilize training.
LayerNorm improves gradient flow.

## **Mini Transformers Model**

In [73]:
class MiniTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, ffn_hidden)
            for _ in range(num_layers)
        ])

        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)

        attentions = []
        for layer in self.layers:
            x, attn = layer(x)
            attentions.append(attn)

        logits = self.lm_head(x)

        return logits, attentions

## **training Loop**

In [74]:
model = MiniTransformer()

# Define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

def training():
    for step in range(1000):

        inputs = torch.randint(0, vocab_size, (32, seq_len))
        targets = inputs

        logits, _ = model(inputs)

        loss = loss_fn(
            logits.view(-1, vocab_size),
            targets.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 100 == 0:
            print("loss:", loss.item())

## **KV Cache Implementation**

Instead of recomputing K,V for all tokens, store previous K,V during generation.

## **Attention Visualization**

In [ ]:
plt.imshow(attentions[0][0][0].detach().numpy())
plt.title("Attention Map")
plt.colorbar()

## **Inference (Text Generation)**

In [75]:
def generate(model, prompt, steps=20):
    for _ in range(steps):
        logits, _ = model(prompt)
        next_token = torch.argmax(logits[:, -1], dim=-1)

        prompt = torch.cat(
            [prompt, next_token.unsqueeze(-1)],
            dim=1
        )

    return prompt